# 03B — Rebuild Adapter LoRA Terkunci — FINAL V2 CACHE-SAFE

Notebook ini **tidak mencari hyperparameter lagi**. Notebook hanya melatih ulang **satu konfigurasi LoRA final V7** yang sudah dipilih sebelumnya untuk membangun kembali adapter fisik pada instance Vast.ai baru.

## Konfigurasi final yang dikunci dari V7

- Backbone: `meta-llama/Llama-3.1-8B-Instruct`
- Target modules L3: `q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj`
- `r = 16`
- `alpha = 32`
- `learning_rate = 2e-4`
- `dropout = 0.00`
- BF16, **tanpa kuantisasi base model**
- `max_sequence_length = 640`
- Train: **417** pasangan
- Validation: **53** pasangan
- Training seed: **42**
- Training: **4 epoch penuh**
- Checkpoint final yang sudah dikunci V7: **`checkpoint-159` / epoch 3**

## Perbaikan V2

Versi ini secara eksplisit menggunakan cache Hugging Face yang sudah ada di:

`/workspace/.hf_home/hub`

Semua pemuatan tokenizer dan backbone memakai:

- `cache_dir = /workspace/.hf_home/hub`
- `local_files_only = True`

Dengan demikian notebook **tidak akan mengunduh ulang Llama**. Jika cache lokal tidak lengkap, notebook berhenti dengan error yang jelas daripada memulai download puluhan GB.

Section 1 hanya melakukan **audit dependency** dan tidak menjalankan `pip install`, sehingga tidak memicu restart kernel berulang.

Output utama:

- `Artifacts/lora_locked_adapter/`
- `Config/lora_locked_config.json`
- `Hasil/config_selection/lora/selected_lora_config.json`
- `Hasil/final_training/lora_locked_retrain_v2/`


## Dasar lock

Konfigurasi final berasal dari notebook `03_lora_config_validation_FINAL_V7_SEQ640_GPU_SAFE`, yang sebelumnya menyelesaikan pencarian bertahap Stage A–D. Hasil V7 memilih L3, rank 16, alpha 32, learning rate `2e-4`, dropout `0.00`, dan `checkpoint-159` pada epoch 3.

Notebook ini dibuat khusus untuk **memulihkan adapter final pada instance baru**, bukan mengulang pencarian konfigurasi.


## 1. Instalasi dependensi

In [1]:
# ============================================================
# DEPENDENCY AUDIT — TIDAK MENGUBAH ENVIRONMENT
# ============================================================
from importlib.metadata import version, PackageNotFoundError
from packaging.version import Version

EXPECTED_EXACT = {
    "transformers": "4.48.1",
    "accelerate": "1.2.1",
    "huggingface_hub": "0.27.1",
    "peft": "0.14.0",
    "bitsandbytes": "0.48.1",
    "sacrebleu": "2.5.1",
    "sentencepiece": "0.2.0",
}

errors = []

for package_name, expected in EXPECTED_EXACT.items():
    try:
        actual = version(package_name)
    except PackageNotFoundError:
        errors.append(f"{package_name}: NOT INSTALLED")
        continue

    ok = Version(actual) == Version(expected)
    print(f"{package_name:18} {actual:12} {'OK' if ok else 'MISMATCH'}")

    if not ok:
        errors.append(
            f"{package_name}: actual={actual}, expected={expected}"
        )

if errors:
    raise RuntimeError(
        "Dependency audit gagal:\n- "
        + "\n- ".join(errors)
        + "\n\nEnvironment instance ini berbeda dari environment V7. "
          "Jangan lanjutkan training sebelum versi paket disamakan."
    )

print("\nDEPENDENCY AUDIT: PASS")
print("Tidak ada pip install dan tidak perlu restart kernel dari Section 1.")


transformers       4.48.1       OK
accelerate         1.2.1        OK
huggingface_hub    0.27.1       OK
peft               0.14.0       OK
bitsandbytes       0.48.1       OK
sacrebleu          2.5.1        OK
sentencepiece      0.2.0        OK

DEPENDENCY AUDIT: PASS
Tidak ada pip install dan tidak perlu restart kernel dari Section 1.


## 2. Environment, autentikasi, dan GPU preflight

In [2]:
# ============================================================
# ENVIRONMENT + PREFLIGHT
# ============================================================
from __future__ import annotations

import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import re
import json
import time
import math
import random
import hashlib
import traceback
import shutil
import importlib.metadata as importlib_metadata

from pathlib import Path
from datetime import datetime, timezone
from typing import Any

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from packaging.version import Version
from tqdm.auto import tqdm
from huggingface_hub import get_token, HfApi
from sacrebleu.metrics import BLEU, CHRF

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Trainer,
    TrainingArguments,
    set_seed,
)
from transformers.trainer_utils import get_last_checkpoint

from peft import (
    LoraConfig,
    PeftModel,
    TaskType,
    get_peft_model,
)

EXPECTED_VERSIONS = {
    "transformers": "4.48.1",
    "peft": "0.14.0",
    "bitsandbytes": "0.48.1",
}

for package_name, expected_version in EXPECTED_VERSIONS.items():
    actual = importlib_metadata.version(package_name)
    if Version(actual) != Version(expected_version):
        raise RuntimeError(
            f"{package_name}={actual}, tetapi notebook mengunci {expected_version}. "
            "Jalankan Section 1, restart kernel bila diminta, kemudian mulai dari Section 2."
        )

PROJECT_ROOT = Path(
    os.getenv(
        "SKRIPSI_PROJECT_ROOT",
        "/workspace/10122233_CikitaMeydilvira_Skripsi",
    )
).expanduser().resolve()

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        f"PROJECT_ROOT tidak ditemukan: {PROJECT_ROOT}"
    )

HF_HOME = Path(
    os.getenv(
        "HF_HOME",
        "/workspace/.hf_home",
    )
).expanduser().resolve()

HF_HOME.mkdir(
    parents=True,
    exist_ok=True,
)

HF_HUB_CACHE = (
    HF_HOME
    / "hub"
).resolve()

HF_HUB_CACHE.mkdir(
    parents=True,
    exist_ok=True,
)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HUB_CACHE)

DEVICE_INDEX = 0

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA tidak tersedia."
    )

torch.cuda.set_device(
    DEVICE_INDEX
)

if not torch.cuda.is_bf16_supported():
    raise RuntimeError(
        "GPU tidak mendukung BF16. "
        "Notebook ini tidak memiliki fallback FP16/QLoRA."
    )


DTYPE = torch.bfloat16

# ------------------------------------------------------------
# GPU preflight awal.
# Training helper juga mengecek VRAM kembali pada setiap trial.
# ------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()

_initial_free_b, _initial_total_b = torch.cuda.mem_get_info(DEVICE_INDEX)
_initial_free_gib = _initial_free_b / (1024 ** 3)
_initial_total_gib = _initial_total_b / (1024 ** 3)

print(
    "GPU preflight  :",
    f"{_initial_free_gib:.2f} GiB free / {_initial_total_gib:.2f} GiB total",
)

if _initial_free_gib < 20.0:
    raise RuntimeError(
        f"VRAM bebas hanya {_initial_free_gib:.2f} GiB. "
        "Shutdown kernel FSP/Zero-Shot atau proses GPU lain sebelum menjalankan LoRA. "
        "Notebook LoRA memerlukan GPU hampir kosong pada awal trial."
    )

HF_TOKEN = (
    get_token()
    or os.getenv("HF_TOKEN", "").strip()
    or None
)

if not HF_TOKEN:
    raise RuntimeError(
        "Token Hugging Face tidak ditemukan. "
        "Jalankan `hf auth login` di terminal Vast.ai."
    )

EXPECTED_MODEL_ID = (
    "meta-llama/Llama-3.1-8B-Instruct"
)


def resolve_selected_model_id() -> str:

    env_value = os.getenv(
        "SELECTED_TRANSLATOR_MODEL",
        "",
    ).strip()

    if env_value:
        return env_value

    cfg_path = (
        PROJECT_ROOT
        / "Config"
        / "selected_backbone.json"
    )

    if cfg_path.exists():

        with open(
            cfg_path,
            "r",
            encoding="utf-8",
        ) as f:

            obj = json.load(f)

        candidate = str(
            obj.get(
                "model_id",
                "",
            )
        ).strip()

        if candidate:
            return candidate

    return EXPECTED_MODEL_ID


MODEL_ID = resolve_selected_model_id()

if MODEL_ID != EXPECTED_MODEL_ID:
    raise RuntimeError(
        f"Backbone eksperimen utama harus {EXPECTED_MODEL_ID}, "
        f"tetapi ditemukan {MODEL_ID}."
    )

# Validasi akses gated model hanya sebagai audit ringan.
# Training sendiri dipaksa menggunakan cache lokal dan tidak membutuhkan download ulang.
try:
    HfApi(
        token=HF_TOKEN
    ).model_info(
        MODEL_ID
    )
    print("HF gated access : OK")
except Exception as exc:
    print(
        "HF gated access : WARNING — pemeriksaan online gagal, "
        "tetapi notebook akan tetap mencoba cache lokal.",
        type(exc).__name__,
        str(exc)[:200],
    )

MODEL_CACHE_DIR = (
    HF_HUB_CACHE
    / "models--meta-llama--Llama-3.1-8B-Instruct"
)

if not MODEL_CACHE_DIR.exists():
    raise FileNotFoundError(
        "Cache Llama lokal tidak ditemukan: "
        f"{MODEL_CACHE_DIR}. "
        "Notebook V2 sengaja tidak mengunduh ulang model."
    )

cache_size_bytes = sum(
    p.stat().st_size
    for p in MODEL_CACHE_DIR.rglob("*")
    if p.is_file()
)

cache_size_gib = cache_size_bytes / (1024 ** 3)

print(
    "Llama cache     :",
    MODEL_CACHE_DIR,
)
print(
    "Llama cache GiB :",
    f"{cache_size_gib:.2f}",
)

# Llama-3.1-8B-Instruct BF16 cache normalnya belasan GiB.
# Ambang 10 GiB hanya dipakai untuk menangkap cache parsial yang jelas tidak lengkap.
if cache_size_gib < 10.0:
    raise RuntimeError(
        f"Cache Llama tampak tidak lengkap ({cache_size_gib:.2f} GiB). "
        "Notebook dihentikan agar tidak melakukan download ulang."
    )

TRAINING_SEED = 42

set_seed(
    TRAINING_SEED
)

random.seed(
    TRAINING_SEED
)

np.random.seed(
    TRAINING_SEED
)

torch.manual_seed(
    TRAINING_SEED
)

print("PROJECT_ROOT :", PROJECT_ROOT)
print("HF_HOME      :", HF_HOME)
print("HF_HUB_CACHE :", HF_HUB_CACHE)
print("MODEL_ID     :", MODEL_ID)
print("GPU          :", torch.cuda.get_device_name(DEVICE_INDEX))
print(
    "VRAM GiB     :",
    round(
        torch.cuda.get_device_properties(
            DEVICE_INDEX
        ).total_memory / (1024 ** 3),
        2,
    ),
)
print("BF16         :", torch.cuda.is_bf16_supported())
print("Transformers :", importlib_metadata.version("transformers"))
print("PEFT         :", importlib_metadata.version("peft"))
print("bitsandbytes :", importlib_metadata.version("bitsandbytes"))


GPU preflight  : 23.13 GiB free / 23.52 GiB total
HF gated access : OK
Llama cache     : /workspace/.hf_home/hub/models--meta-llama--Llama-3.1-8B-Instruct
Llama cache GiB : 29.93
PROJECT_ROOT : /workspace/10122233_CikitaMeydilvira_Skripsi
HF_HOME      : /workspace/.hf_home
HF_HUB_CACHE : /workspace/.hf_home/hub
MODEL_ID     : meta-llama/Llama-3.1-8B-Instruct
GPU          : NVIDIA GeForce RTX 4090
VRAM GiB     : 23.52
BF16         : True
Transformers : 4.48.1
PEFT         : 0.14.0
bitsandbytes : 0.48.1


## 3. Dataset final — train + validation; test hanya dicek keberadaannya

In [3]:
# ============================================================
# DATASET FINAL — HARD LOCK
# Train = 417 / 44 conversations
# Validation = 53 / 3 conversations
# Test = hanya dicek ada, ISI TIDAK DIBACA
# ============================================================

DATASET_DIR = PROJECT_ROOT / "Dataset"
TRAIN_PATH = DATASET_DIR / "train.csv"
VALID_PATH = DATASET_DIR / "validation.csv"
TEST_PATH = DATASET_DIR / "test.csv"

EXPECTED_TRAIN = 417
EXPECTED_VALID = 53
EXPECTED_TRAIN_CONV = 44
EXPECTED_VALID_CONV = 3

REQUIRED_COLUMNS = [
    "row_id",
    "conversation_no",
    "turn_order",
    "duri_text",
    "indonesia_text",
]

for path in [TRAIN_PATH, VALID_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset final tidak ditemukan: {path}"
        )

def load_parallel_split(
    path: Path,
    expected_rows: int,
) -> pd.DataFrame:

    df = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        encoding="utf-8-sig",
    )

    df.columns = [
        str(col).strip()
        for col in df.columns
    ]

    missing = [
        col
        for col in REQUIRED_COLUMNS
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{path.name}: kolom wajib hilang: {missing}"
        )

    df = df[
        REQUIRED_COLUMNS
    ].copy()

    for col in REQUIRED_COLUMNS:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
        )

    if len(df) != expected_rows:
        raise ValueError(
            f"{path.name}: diharapkan {expected_rows} baris, "
            f"ditemukan {len(df)}."
        )

    if df["row_id"].eq("").any():
        raise ValueError(
            f"{path.name}: terdapat row_id kosong."
        )

    if df["row_id"].duplicated().any():
        bad = (
            df.loc[
                df["row_id"].duplicated(keep=False),
                "row_id",
            ]
            .tolist()[:10]
        )
        raise ValueError(
            f"{path.name}: row_id duplikat. Contoh: {bad}"
        )

    if (
        df["duri_text"].eq("").any()
        or df["indonesia_text"].eq("").any()
    ):
        raise ValueError(
            f"{path.name}: terdapat teks kosong."
        )

    return df.reset_index(drop=True)

def norm_for_overlap(text: str) -> str:
    return re.sub(
        r"\s+",
        " ",
        str(text).strip().casefold(),
    )

train_df = load_parallel_split(
    TRAIN_PATH,
    EXPECTED_TRAIN,
)

valid_df = load_parallel_split(
    VALID_PATH,
    EXPECTED_VALID,
)

# ------------------------------------------------------------
# Conversation-level split audit
# ------------------------------------------------------------
train_conversations = set(
    train_df["conversation_no"]
)

valid_conversations = set(
    valid_df["conversation_no"]
)

if len(train_conversations) != EXPECTED_TRAIN_CONV:
    raise ValueError(
        f"Train conversation diharapkan {EXPECTED_TRAIN_CONV}, "
        f"ditemukan {len(train_conversations)}."
    )

if len(valid_conversations) != EXPECTED_VALID_CONV:
    raise ValueError(
        f"Validation conversation diharapkan {EXPECTED_VALID_CONV}, "
        f"ditemukan {len(valid_conversations)}."
    )

conversation_overlap = sorted(
    train_conversations
    & valid_conversations
)

if conversation_overlap:
    raise RuntimeError(
        "FATAL: conversation leakage train-validation: "
        f"{conversation_overlap}"
    )

# ------------------------------------------------------------
# Source overlap audit
# Source identik tidak otomatis fatal:
# recurring utterance dapat muncul pada conversation berbeda.
# Exact Duri–Indonesia pair lintas split tetap fatal.
# ------------------------------------------------------------
train_audit = train_df.copy()
valid_audit = valid_df.copy()

train_audit["_src_norm"] = (
    train_audit["duri_text"]
    .map(norm_for_overlap)
)
train_audit["_tgt_norm"] = (
    train_audit["indonesia_text"]
    .map(norm_for_overlap)
)

valid_audit["_src_norm"] = (
    valid_audit["duri_text"]
    .map(norm_for_overlap)
)
valid_audit["_tgt_norm"] = (
    valid_audit["indonesia_text"]
    .map(norm_for_overlap)
)

source_overlap = sorted(
    set(train_audit["_src_norm"])
    & set(valid_audit["_src_norm"])
)

SOURCE_OVERLAP_COUNT = len(
    source_overlap
)

source_overlap_detail = pd.DataFrame()

print(
    "SOURCE OVERLAP TRAIN–VALIDATION :",
    SOURCE_OVERLAP_COUNT,
)

if source_overlap:

    source_overlap_detail = (
        train_audit[
            train_audit["_src_norm"].isin(
                source_overlap
            )
        ][
            [
                "row_id",
                "conversation_no",
                "turn_order",
                "duri_text",
                "indonesia_text",
                "_src_norm",
                "_tgt_norm",
            ]
        ]
        .merge(
            valid_audit[
                valid_audit["_src_norm"].isin(
                    source_overlap
                )
            ][
                [
                    "row_id",
                    "conversation_no",
                    "turn_order",
                    "duri_text",
                    "indonesia_text",
                    "_src_norm",
                    "_tgt_norm",
                ]
            ],
            on="_src_norm",
            how="inner",
            suffixes=("_train", "_valid"),
        )
    )

    source_overlap_detail["target_identik"] = (
        source_overlap_detail[
            "_tgt_norm_train"
        ]
        ==
        source_overlap_detail[
            "_tgt_norm_valid"
        ]
    )

    print(
        "WARNING: ditemukan source Duri identik lintas split. "
        "Ini dicatat sebagai recurring source dan belum otomatis "
        "merupakan exact-pair leakage."
    )

    display(
        source_overlap_detail[
            [
                "_src_norm",
                "row_id_train",
                "conversation_no_train",
                "duri_text_train",
                "indonesia_text_train",
                "row_id_valid",
                "conversation_no_valid",
                "duri_text_valid",
                "indonesia_text_valid",
                "target_identik",
            ]
        ]
    )
else:
    print(
        "OK: tidak ada source identik train-validation."
    )

train_pairs = set(
    zip(
        train_audit["_src_norm"],
        train_audit["_tgt_norm"],
    )
)

valid_pairs = set(
    zip(
        valid_audit["_src_norm"],
        valid_audit["_tgt_norm"],
    )
)

exact_pair_overlap = sorted(
    train_pairs
    & valid_pairs
)

EXACT_PAIR_OVERLAP_COUNT = len(
    exact_pair_overlap
)

print(
    "EXACT PAIR OVERLAP TRAIN–VALIDATION :",
    EXACT_PAIR_OVERLAP_COUNT,
)

if exact_pair_overlap:

    if not source_overlap_detail.empty:

        exact_detail = (
            source_overlap_detail[
                source_overlap_detail[
                    "target_identik"
                ]
            ]
            .copy()
        )

        if not exact_detail.empty:
            display(
                exact_detail[
                    [
                        "_src_norm",
                        "row_id_train",
                        "conversation_no_train",
                        "indonesia_text_train",
                        "row_id_valid",
                        "conversation_no_valid",
                        "indonesia_text_valid",
                    ]
                ]
            )

    raise RuntimeError(
        "FATAL: terdapat pasangan Duri–Indonesia identik "
        "antara train dan validation. Split harus diperbaiki."
    )

print(
    "PASS: tidak ada exact Duri–Indonesia pair "
    "train-validation."
)

if SOURCE_OVERLAP_COUNT > 0:
    print(
        "CATATAN: recurring source lintas conversation akan "
        "dicatat pada output audit. Untuk LoRA tidak ada retrieval "
        "yang dapat difilter; split conversation tetap dipertahankan."
    )

# ------------------------------------------------------------
# Dataset fingerprint
# ------------------------------------------------------------
def dataset_fingerprint() -> str:

    h = hashlib.sha256()

    for split_name, df in [
        ("train", train_df),
        ("validation", valid_df),
    ]:

        ordered = df.sort_values(
            "row_id",
            kind="stable",
        )

        for row in ordered.itertuples(
            index=False
        ):

            payload = {
                "split": split_name,
                "row_id": row.row_id,
                "conversation_no": row.conversation_no,
                "turn_order": row.turn_order,
                "duri_text": row.duri_text,
                "indonesia_text": row.indonesia_text,
            }

            h.update(
                (
                    json.dumps(
                        payload,
                        ensure_ascii=False,
                        sort_keys=True,
                        separators=(",", ":"),
                    )
                    + "\n"
                ).encode("utf-8")
            )

    return h.hexdigest()

DATASET_FINGERPRINT = (
    dataset_fingerprint()
)

print()
print("Train path          :", TRAIN_PATH)
print("Validation path     :", VALID_PATH)
print("Train               :", len(train_df), "/", len(train_conversations), "conversation")
print("Validation          :", len(valid_df), "/", len(valid_conversations), "conversation")
print("Test                : TIDAK DIBACA")
print("Dataset fingerprint :", DATASET_FINGERPRINT)


SOURCE OVERLAP TRAIN–VALIDATION : 1


,_src_norm,row_id_train,conversation_no_train,duri_text_train,indonesia_text_train,row_id_valid,conversation_no_valid,duri_text_valid,indonesia_text_valid,target_identik
0,waqdingraka ke undinaq?,C01_T05_A3,1,Waqdingraka ke undinaq?,Bolehkah jika saya ikut?,C40_T03_A2,40,Waqdingraka ke undinaq?,Bolehkah saya ikut?,False


EXACT PAIR OVERLAP TRAIN–VALIDATION : 0
PASS: tidak ada exact Duri–Indonesia pair train-validation.
CATATAN: recurring source lintas conversation akan dicatat pada output audit. Untuk LoRA tidak ada retrieval yang dapat difilter; split conversation tetap dipertahankan.

Train path          : /workspace/10122233_CikitaMeydilvira_Skripsi/Dataset/train.csv
Validation path     : /workspace/10122233_CikitaMeydilvira_Skripsi/Dataset/validation.csv
Train               : 417 / 44 conversation
Validation          : 53 / 3 conversation
Test                : TIDAK DIBACA
Dataset fingerprint : 6c8f3cb81c2dfe100390e26ca33147c8324e90dffa31092ae63c18310ba6fdcc


## 4. Tokenizer, prompt, sequence-length audit, dan assistant-only loss

In [4]:
# ============================================================
# TOKENIZER + DATASET SFT
# ============================================================
from torch.utils.data import Dataset

SYSTEM_MESSAGE = (
    "Anda adalah penerjemah bahasa Duri ke bahasa Indonesia."
)

USER_TEMPLATE = (
    "Terjemahkan teks bahasa Duri berikut ke bahasa Indonesia.\n"
    "Keluarkan hanya hasil terjemahan tanpa penjelasan tambahan.\n\n"
    "Bahasa Duri: {duri_text}\n"
    "Bahasa Indonesia:"
)

MAX_SEQUENCE_LENGTH = 640
VALIDATION_MAX_NEW_TOKENS = 128

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    use_fast=True,
    trust_remote_code=False,
    cache_dir=str(HF_HUB_CACHE),
    local_files_only=True,
)

if tokenizer.chat_template is None:
    raise RuntimeError(
        "Tokenizer tidak memiliki chat_template."
    )

if tokenizer.pad_token_id is None:

    if tokenizer.eos_token_id is None:
        raise RuntimeError(
            "Tokenizer tidak memiliki pad_token_id maupun eos_token_id."
        )

    tokenizer.pad_token = (
        tokenizer.eos_token
    )

tokenizer.padding_side = (
    "right"
)


def build_user_messages(
    duri_text: str,
) -> list[dict[str, str]]:

    return [
        {
            "role": "system",
            "content": SYSTEM_MESSAGE,
        },
        {
            "role": "user",
            "content": USER_TEMPLATE.format(
                duri_text=duri_text
            ),
        },
    ]


def build_training_messages(
    duri_text: str,
    indonesia_text: str,
) -> list[dict[str, str]]:

    return (
        build_user_messages(
            duri_text
        )
        + [
            {
                "role": "assistant",
                "content": indonesia_text,
            }
        ]
    )


def encode_sft(
    duri_text: str,
    indonesia_text: str,
) -> dict[str, list[int]]:

    prompt_ids = (
        tokenizer.apply_chat_template(
            build_user_messages(
                duri_text
            ),
            tokenize=True,
            add_generation_prompt=True,
        )
    )

    full_ids = (
        tokenizer.apply_chat_template(
            build_training_messages(
                duri_text,
                indonesia_text,
            ),
            tokenize=True,
            add_generation_prompt=False,
        )
    )

    common = 0

    for prompt_token, full_token in zip(
        prompt_ids,
        full_ids,
    ):

        if prompt_token != full_token:
            break

        common += 1

    if len(full_ids) > MAX_SEQUENCE_LENGTH:
        raise ValueError(
            f"Sampel > {MAX_SEQUENCE_LENGTH} token. "
            f"Duri={duri_text!r}, panjang={len(full_ids)}"
        )

    labels = (
        [-100] * common
        + list(
            full_ids[
                common:
            ]
        )
    )

    if (
        not labels
        or all(
            token_id == -100
            for token_id in labels
        )
    ):
        raise ValueError(
            "Tidak ada token assistant untuk loss."
        )

    return {
        "input_ids": list(
            full_ids
        ),
        "attention_mask": [
            1
        ] * len(
            full_ids
        ),
        "labels": labels,
    }


# ------------------------------------------------------------
# Audit panjang seluruh train + validation
# ------------------------------------------------------------

length_rows = []

for split_name, data in [
    (
        "train",
        train_df,
    ),
    (
        "validation",
        valid_df,
    ),
]:

    for row in data.itertuples(
        index=False
    ):

        full_ids = (
            tokenizer.apply_chat_template(
                build_training_messages(
                    row.duri_text,
                    row.indonesia_text,
                ),
                tokenize=True,
                add_generation_prompt=False,
            )
        )

        length_rows.append(
            {
                "split": split_name,
                "row_id": row.row_id,
                "tokens": len(
                    full_ids
                ),
            }
        )

length_audit_df = pd.DataFrame(
    length_rows
)

print(
    "Sequence min  :",
    int(
        length_audit_df[
            "tokens"
        ].min()
    ),
)

print(
    "Sequence mean :",
    round(
        float(
            length_audit_df[
                "tokens"
            ].mean()
        ),
        2,
    ),
)

print(
    "Sequence max  :",
    int(
        length_audit_df[
            "tokens"
        ].max()
    ),
)

too_long = (
    length_audit_df[
        "tokens"
    ]
    > MAX_SEQUENCE_LENGTH
)

print(
    f">{MAX_SEQUENCE_LENGTH:<12}:",
    int(
        too_long.sum()
    ),
)


# Informasi baris terpanjang untuk audit/reproducibility.
_longest_rows = (
    length_audit_df
    .sort_values(
        ["tokens", "split", "row_id"],
        ascending=[False, True, True],
        kind="stable",
    )
    .head(10)
)

print("\\n10 sequence terpanjang:")
display(_longest_rows)

if too_long.any():

    too_long_df = (
        length_audit_df[
            too_long
        ]
        .sort_values(
            "tokens",
            ascending=False,
        )
        .copy()
    )

    display(
        too_long_df
    )

    observed_max = int(
        too_long_df[
            "tokens"
        ].max()
    )

    raise ValueError(
        "Ada sampel yang melebihi MAX_SEQUENCE_LENGTH="
        f"{MAX_SEQUENCE_LENGTH}. Panjang maksimum teramati="
        f"{observed_max}. Notebook sengaja tidak melakukan truncation "
        "source/reference secara diam-diam. Naikkan batas secara eksplisit "
        "dan gunakan nilai yang sama untuk seluruh trial."
    )

print(
    "PASS sequence-length audit: seluruh train+validation <=",
    MAX_SEQUENCE_LENGTH,
    "token; tidak ada truncation."
)


class SFTDataset(
    Dataset
):

    def __init__(
        self,
        dataframe: pd.DataFrame,
        split_name: str,
    ) -> None:

        self.items = [
            encode_sft(
                row.duri_text,
                row.indonesia_text,
            )
            for row in tqdm(
                dataframe.itertuples(
                    index=False
                ),
                total=len(
                    dataframe
                ),
                desc=(
                    f"Tokenisasi {split_name}"
                ),
            )
        ]

    def __len__(
        self,
    ) -> int:

        return len(
            self.items
        )

    def __getitem__(
        self,
        index: int,
    ) -> dict[str, list[int]]:

        return self.items[
            index
        ]


class AssistantOnlyCollator:

    def __init__(
        self,
        pad_token_id: int,
        pad_to_multiple_of: int = 8,
    ) -> None:

        self.pad_token_id = int(
            pad_token_id
        )

        self.pad_to_multiple_of = int(
            pad_to_multiple_of
        )

    def __call__(
        self,
        features: list[
            dict[str, list[int]]
        ],
    ) -> dict[str, torch.Tensor]:

        max_len = max(
            len(
                item[
                    "input_ids"
                ]
            )
            for item in features
        )

        if self.pad_to_multiple_of > 1:

            max_len = (
                (
                    max_len
                    + self.pad_to_multiple_of
                    - 1
                )
                // self.pad_to_multiple_of
                * self.pad_to_multiple_of
            )

        input_ids = []
        attention_masks = []
        labels = []

        for item in features:

            pad = (
                max_len
                - len(
                    item[
                        "input_ids"
                    ]
                )
            )

            input_ids.append(
                item[
                    "input_ids"
                ]
                + [
                    self.pad_token_id
                ] * pad
            )

            attention_masks.append(
                item[
                    "attention_mask"
                ]
                + [
                    0
                ] * pad
            )

            labels.append(
                item[
                    "labels"
                ]
                + [
                    -100
                ] * pad
            )

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long,
            ),
            "attention_mask": torch.tensor(
                attention_masks,
                dtype=torch.long,
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long,
            ),
        }


train_dataset = SFTDataset(
    train_df,
    "train",
)

valid_dataset = SFTDataset(
    valid_df,
    "validation",
)

collator = AssistantOnlyCollator(
    tokenizer.pad_token_id
)

print(
    "Dataset tokenized:",
    len(train_dataset),
    len(valid_dataset),
)


Sequence min  : 99
Sequence mean : 128.72
Sequence max  : 599
>640         : 0
\n10 sequence terpanjang:


,split,row_id,tokens
222,train,C28_T04_B2,599
192,train,C24_T06_B3,284
406,train,C49_T04_B2,271
289,train,C36_T10_B5,268
190,train,C24_T04_B2,252
400,train,C48_T08_B4,248
463,validation,C40_T26_B13,247
321,train,C39_T12_B6,216
145,train,C19_T08_B4,213
325,train,C41_T04_B2,213


PASS sequence-length audit: seluruh train+validation <= 640 token; tidak ada truncation.


Tokenisasi train:   0%|          | 0/417 [00:00<?, ?it/s]

Tokenisasi validation:   0%|          | 0/53 [00:00<?, ?it/s]

Dataset tokenized: 417 53


## 5. Protokol training yang sama dengan V7

Cell berikut mempertahankan konstanta dan helper V7 agar format prompt, assistant-only loss, optimizer, scheduler, checkpoint tiap epoch, validasi deterministik, dan prosedur cleanup GPU tetap sama.

Walaupun struktur helper masih memuat fungsi pencarian, notebook ini **hanya memanggil satu trial final**.


In [ ]:
# ============================================================
# SEARCH SPACE — FINAL V6
# ============================================================

TARGET_MODULE_OPTIONS = {
    "L1": [
        "q_proj",
        "v_proj",
    ],
    "L2": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    "L3": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
}

RANK_VALUES = [
    4,
    8,
    16,
]

LEARNING_RATE_VALUES = [
    5e-5,
    1e-4,
    2e-4,
]

DROPOUT_VALUES = [
    0.00,
    0.05,
    0.10,
]

# ------------------------------------------------------------
# Controlled anchor untuk Stage A
# ------------------------------------------------------------
BASE_RANK = 16
BASE_ALPHA = 32
BASE_LEARNING_RATE = 1e-4
BASE_DROPOUT = 0.05

# Setelah Stage A, alpha mengikuti rank:
# alpha/r = 2
ALPHA_MULTIPLIER = 2

LR_SCHEDULER_TYPE = "linear"
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 1.0

PER_DEVICE_TRAIN_BATCH_SIZE = 1
PER_DEVICE_EVAL_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8

MAX_EPOCHS = 4

# Semua checkpoint akhir epoch disimpan agar setiap epoch
# dapat dievaluasi dengan chrF++/BLEU.
SAVE_TOTAL_LIMIT = MAX_EPOCHS

OPTIMIZER = "paged_adamw_8bit"

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "Hasil"
    / "config_selection"
    / "lora_target_first_v7"
)

TRIAL_ROOT = (
    OUTPUT_ROOT
    / "trials"
)

STAGE_ROOT = (
    OUTPUT_ROOT
    / "stages"
)

FIGURE_ROOT = (
    OUTPUT_ROOT
    / "figures_bab4"
)

for directory in [
    OUTPUT_ROOT,
    TRIAL_ROOT,
    STAGE_ROOT,
    FIGURE_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

# Canonical outputs
CANONICAL_SELECTED_DIR = (
    PROJECT_ROOT
    / "Hasil"
    / "config_selection"
    / "lora"
)

CANONICAL_SELECTED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CANONICAL_SELECTED_PATH = (
    CANONICAL_SELECTED_DIR
    / "selected_lora_config.json"
)

CONFIG_LOCK_PATH = (
    PROJECT_ROOT
    / "Config"
    / "lora_locked_config.json"
)

LOCKED_ADAPTER_DIR = (
    PROJECT_ROOT
    / "Artifacts"
    / "lora_locked_adapter"
)

CHECKPOINT_SELECTION_RULE = (
    "complete validation -> chrF++ descending; "
    "BLEU descending; validation loss ascending; "
    "epoch ascending"
)

STAGE_SELECTION_RULE = (
    "complete validation -> chrF++ descending; "
    "BLEU descending; selected checkpoint validation loss ascending; "
    "trainable parameters ascending"
)

print("OUTPUT_ROOT :", OUTPUT_ROOT)
print("Stage A     : target_modules L1/L2/L3, fixed r=16 alpha=32 LR=1e-4 dropout=0.05")
print("Stage B     : rank", RANK_VALUES)
print("Stage C LR  :", LEARNING_RATE_VALUES)
print("Stage D drop:", DROPOUT_VALUES)
print("Epoch       :", MAX_EPOCHS)
print("Early stop  : TIDAK DIGUNAKAN")
print("Checkpoint  :", CHECKPOINT_SELECTION_RULE)


OUTPUT_ROOT : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/config_selection/lora_target_first_v7
Stage A     : target_modules L1/L2/L3, fixed r=16 alpha=32 LR=1e-4 dropout=0.05
Stage B     : rank [4, 8, 16]
Stage C LR  : [5e-05, 0.0001, 0.0002]
Stage D drop: [0.0, 0.05, 0.1]
Epoch       : 4
Early stop  : TIDAK DIGUNAKAN
Checkpoint  : complete validation -> chrF++ descending; BLEU descending; validation loss ascending; epoch ascending


## 6. Helper training dan validation — implementasi V7

In [6]:
# ============================================================
# HELPER TRAINING + CHECKPOINT SELECTION
# ============================================================

def gpu_sync() -> None:

    if torch.cuda.is_available():
        torch.cuda.synchronize(
            DEVICE_INDEX
        )


def clear_gpu_strong() -> None:

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

        try:
            torch.cuda.synchronize(
                DEVICE_INDEX
            )
        except Exception:
            pass

    gc.collect()


def free_vram_gib() -> float:

    clear_gpu_strong()

    free_bytes, _ = (
        torch.cuda.mem_get_info(
            DEVICE_INDEX
        )
    )

    return float(
        free_bytes
        / (1024 ** 3)
    )


def clean_translation_output(
    text: str,
) -> str:

    value = str(
        text
    ).strip()

    prefixes = [
        "Bahasa Indonesia:",
        "Terjemahan:",
        "Jawaban:",
    ]

    changed = True

    while changed:

        changed = False

        for prefix in prefixes:

            if value.lower().startswith(
                prefix.lower()
            ):

                value = value[
                    len(prefix):
                ].strip()

                changed = True

    return value.strip()


def generate_validation_one(
    model,
    duri_text: str,
) -> str:

    messages = build_user_messages(
        duri_text
    )

    encoded = (
        tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
        )
    )

    encoded = {
        key: value.to(
            model.device
        )
        for key, value in encoded.items()
    }

    terminators = []

    if tokenizer.eos_token_id is not None:

        terminators.append(
            int(
                tokenizer.eos_token_id
            )
        )

    eot = (
        tokenizer.convert_tokens_to_ids(
            "<|eot_id|>"
        )
    )

    if (
        isinstance(
            eot,
            int,
        )
        and eot >= 0
        and eot
        != tokenizer.unk_token_id
        and eot
        not in terminators
    ):

        terminators.append(
            eot
        )

    generation_kwargs = {
        "max_new_tokens":
            VALIDATION_MAX_NEW_TOKENS,
        "do_sample":
            False,
        "num_beams":
            1,
        "repetition_penalty":
            1.0,
        "use_cache":
            True,
        "pad_token_id":
            tokenizer.pad_token_id,
    }

    if terminators:

        generation_kwargs[
            "eos_token_id"
        ] = terminators

    with torch.inference_mode():

        outputs = model.generate(
            **encoded,
            **generation_kwargs,
        )

    prompt_tokens = int(
        encoded[
            "input_ids"
        ].shape[
            -1
        ]
    )

    generated = outputs[
        0,
        prompt_tokens:
    ]

    decoded = tokenizer.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    return clean_translation_output(
        decoded
    )


def deterministic_validation(
    model,
    evaluation_id: str,
) -> tuple[
    pd.DataFrame,
    float,
    float,
]:

    model.eval()
    model.config.use_cache = True

    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass

    records = []

    for row in tqdm(
        valid_df.itertuples(
            index=False
        ),
        total=len(
            valid_df
        ),
        desc=(
            f"Validation {evaluation_id}"
        ),
    ):

        start = (
            time.perf_counter()
        )

        output = (
            generate_validation_one(
                model,
                row.duri_text,
            )
        )

        elapsed = (
            time.perf_counter()
            - start
        )

        if not output:

            raise RuntimeError(
                f"{evaluation_id}: output kosong "
                f"untuk row_id={row.row_id}"
            )

        records.append(
            {
                "evaluation_id":
                    evaluation_id,
                "row_id":
                    row.row_id,
                "source_text":
                    row.duri_text,
                "reference_text":
                    row.indonesia_text,
                "selected_output":
                    output,
                "generation_time_seconds":
                    elapsed,
            }
        )

    pred_df = pd.DataFrame(
        records
    )

    hypotheses = (
        pred_df[
            "selected_output"
        ].tolist()
    )

    references = (
        pred_df[
            "reference_text"
        ].tolist()
    )

    bleu = float(
        BLEU(
            tokenize="13a"
        ).corpus_score(
            hypotheses,
            [
                references
            ],
        ).score
    )

    chrfpp = float(
        CHRF(
            word_order=2
        ).corpus_score(
            hypotheses,
            [
                references
            ],
        ).score
    )

    return (
        pred_df,
        bleu,
        chrfpp,
    )


def canonical_trial_payload(
    *,
    rank: int,
    target_label: str,
    learning_rate: float,
    dropout: float,
) -> dict[str, Any]:

    if target_label not in TARGET_MODULE_OPTIONS:

        raise KeyError(
            f"Target label tidak dikenal: {target_label}"
        )

    alpha = int(
        ALPHA_MULTIPLIER
        * rank
    )

    return {
        "model_id":
            MODEL_ID,
        "dataset_fingerprint":
            DATASET_FINGERPRINT,
        "source_overlap_count":
            int(SOURCE_OVERLAP_COUNT),
        "exact_pair_overlap_count":
            int(EXACT_PAIR_OVERLAP_COUNT),
        "test_read":
            False,
        "prompt_protocol":
            "llama_chat_system_plus_user_translation_v6",
        "rank":
            int(
                rank
            ),
        "alpha":
            alpha,
        "alpha_over_r":
            float(
                alpha
                / rank
            ),
        "dropout":
            float(
                dropout
            ),
        "target_label":
            target_label,
        "target_modules":
            list(
                TARGET_MODULE_OPTIONS[
                    target_label
                ]
            ),
        "learning_rate":
            float(
                learning_rate
            ),
        "lr_scheduler_type":
            LR_SCHEDULER_TYPE,
        "warmup_ratio":
            float(
                WARMUP_RATIO
            ),
        "weight_decay":
            float(
                WEIGHT_DECAY
            ),
        "max_grad_norm":
            float(
                MAX_GRAD_NORM
            ),
        "per_device_train_batch_size":
            int(
                PER_DEVICE_TRAIN_BATCH_SIZE
            ),
        "gradient_accumulation_steps":
            int(
                GRADIENT_ACCUMULATION_STEPS
            ),
        "max_epochs":
            int(
                MAX_EPOCHS
            ),
        "early_stopping":
            False,
        "checkpoint_selection_rule":
            CHECKPOINT_SELECTION_RULE,
        "max_sequence_length":
            int(
                MAX_SEQUENCE_LENGTH
            ),
        "training_seed":
            int(
                TRAINING_SEED
            ),
        "train_rows":
            int(
                len(
                    train_df
                )
            ),
        "validation_rows":
            int(
                len(
                    valid_df
                )
            ),
    }


def payload_fingerprint(
    payload: dict[str, Any],
) -> str:

    raw = json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(
            ",",
            ":",
        ),
    ).encode(
        "utf-8"
    )

    return hashlib.sha256(
        raw
    ).hexdigest()[
        :16
    ]


def load_lora_training_model(
    payload: dict[str, Any],
):

    clear_gpu_strong()

    free_before = (
        free_vram_gib()
    )

    print(
        "VRAM free sebelum load:",
        f"{free_before:.2f} GiB",
    )

    if free_before < 20.0:

        raise RuntimeError(
            f"VRAM bebas hanya {free_before:.2f} GiB. "
            "Kemungkinan model/proses GPU lama masih tertahan. "
            "Restart kernel atau hentikan proses GPU lain."
        )

    base = (
        AutoModelForCausalLM
        .from_pretrained(
            MODEL_ID,
            token=HF_TOKEN,
            torch_dtype=DTYPE,
            device_map={
                "": DEVICE_INDEX
            },
            low_cpu_mem_usage=True,
            trust_remote_code=False,
            cache_dir=str(
                HF_HUB_CACHE
            ),
            local_files_only=True,
        )
    )

    if (
        bool(
            getattr(
                base,
                "is_loaded_in_4bit",
                False,
            )
        )
        or bool(
            getattr(
                base,
                "is_loaded_in_8bit",
                False,
            )
        )
    ):

        raise RuntimeError(
            "Base model terkuantisasi. "
            "Eksperimen harus LoRA BF16, bukan QLoRA."
        )

    target_modules = (
        payload[
            "target_modules"
        ]
    )

    missing_modules = [
        module_name
        for module_name in target_modules
        if not any(
            name.endswith(
                module_name
            )
            for name, _ in base.named_modules()
        )
    ]

    if missing_modules:

        raise RuntimeError(
            "Target module tidak ditemukan: "
            f"{missing_modules}"
        )

    base.config.use_cache = False

    base.gradient_checkpointing_enable()

    base.enable_input_require_grads()

    lora_config = LoraConfig(
        r=int(
            payload[
                "rank"
            ]
        ),
        lora_alpha=int(
            payload[
                "alpha"
            ]
        ),
        lora_dropout=float(
            payload[
                "dropout"
            ]
        ),
        bias="none",
        task_type=(
            TaskType.CAUSAL_LM
        ),
        target_modules=(
            target_modules
        ),
    )

    model = get_peft_model(
        base,
        lora_config,
    )

    trainable = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    total = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    unexpected = [
        name
        for name, parameter
        in model.named_parameters()
        if (
            parameter.requires_grad
            and "lora_" not in name.lower()
        )
    ]

    if unexpected:

        raise RuntimeError(
            "Parameter non-LoRA ikut trainable. "
            f"Contoh: {unexpected[:10]}"
        )

    print(
        f"Trainable: {trainable:,}/{total:,} "
        f"({100 * trainable / total:.4f}%)"
    )

    return (
        model,
        trainable,
        total,
    )


def list_epoch_checkpoints(
    trainer_dir: Path,
) -> list[Path]:

    checkpoints = []

    for path in trainer_dir.glob(
        "checkpoint-*"
    ):

        if not path.is_dir():
            continue

        match = re.fullmatch(
            r"checkpoint-(\d+)",
            path.name,
        )

        if not match:
            continue

        checkpoints.append(
            (
                int(
                    match.group(
                        1
                    )
                ),
                path,
            )
        )

    checkpoints.sort(
        key=lambda item: item[
            0
        ]
    )

    return [
        path
        for _, path in checkpoints
    ]


def eval_loss_for_step(
    history_df: pd.DataFrame,
    step: int,
) -> tuple[
    float,
    float | None,
]:

    if (
        "eval_loss"
        not in history_df.columns
        or "step"
        not in history_df.columns
    ):

        raise RuntimeError(
            "training_log tidak memiliki eval_loss/step."
        )

    eval_rows = history_df[
        history_df[
            "eval_loss"
        ].notna()
    ].copy()

    matched = eval_rows[
        pd.to_numeric(
            eval_rows[
                "step"
            ],
            errors="coerce",
        )
        == int(
            step
        )
    ]

    if matched.empty:

        raise RuntimeError(
            f"eval_loss untuk checkpoint step={step} "
            "tidak ditemukan."
        )

    row = matched.iloc[
        -1
    ]

    eval_loss = float(
        row[
            "eval_loss"
        ]
    )

    epoch = None

    if (
        "epoch"
        in matched.columns
        and pd.notna(
            row[
                "epoch"
            ]
        )
    ):

        epoch = float(
            row[
                "epoch"
            ]
        )

    return (
        eval_loss,
        epoch,
    )


def load_checkpoint_for_inference(
    checkpoint_path: Path,
):

    clear_gpu_strong()

    free_before = (
        free_vram_gib()
    )

    if free_before < 20.0:

        raise RuntimeError(
            f"VRAM bebas hanya {free_before:.2f} GiB "
            "sebelum evaluasi checkpoint."
        )

    base = (
        AutoModelForCausalLM
        .from_pretrained(
            MODEL_ID,
            token=HF_TOKEN,
            torch_dtype=DTYPE,
            device_map={
                "": DEVICE_INDEX
            },
            low_cpu_mem_usage=True,
            trust_remote_code=False,
            cache_dir=str(
                HF_HUB_CACHE
            ),
            local_files_only=True,
        )
    )

    if (
        bool(
            getattr(
                base,
                "is_loaded_in_4bit",
                False,
            )
        )
        or bool(
            getattr(
                base,
                "is_loaded_in_8bit",
                False,
            )
        )
    ):

        raise RuntimeError(
            "Base model evaluasi terkuantisasi."
        )

    model = PeftModel.from_pretrained(
        base,
        str(
            checkpoint_path
        ),
        is_trainable=False,
    )

    model.eval()
    model.config.use_cache = True

    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass

    return model


def evaluate_checkpoint(
    *,
    trial_id: str,
    checkpoint_path: Path,
    history_df: pd.DataFrame,
    checkpoint_eval_root: Path,
) -> dict[str, Any]:

    match = re.fullmatch(
        r"checkpoint-(\d+)",
        checkpoint_path.name,
    )

    if not match:

        raise ValueError(
            f"Nama checkpoint tidak valid: {checkpoint_path}"
        )

    step = int(
        match.group(
            1
        )
    )

    eval_loss, epoch = (
        eval_loss_for_step(
            history_df,
            step,
        )
    )

    output_dir = (
        checkpoint_eval_root
        / checkpoint_path.name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    metrics_path = (
        output_dir
        / "metrics.json"
    )

    predictions_path = (
        output_dir
        / "validation_predictions.csv"
    )

    if (
        metrics_path.exists()
        and predictions_path.exists()
    ):

        with open(
            metrics_path,
            "r",
            encoding="utf-8",
        ) as f:

            metrics = json.load(
                f
            )

        print(
            f"Reuse evaluasi "
            f"{trial_id}/{checkpoint_path.name}"
        )

        return metrics

    model = None

    try:

        print(
            "\nEvaluasi:",
            trial_id,
            checkpoint_path.name,
        )

        model = (
            load_checkpoint_for_inference(
                checkpoint_path
            )
        )

        pred_df, bleu, chrfpp = (
            deterministic_validation(
                model,
                (
                    f"{trial_id}_"
                    f"{checkpoint_path.name}"
                ),
            )
        )

        pred_df.to_csv(
            predictions_path,
            index=False,
            encoding="utf-8-sig",
        )

        metrics = {
            "trial_id":
                trial_id,
            "checkpoint_name":
                checkpoint_path.name,
            "checkpoint_path":
                str(
                    checkpoint_path
                ),
            "step":
                int(
                    step
                ),
            "epoch":
                (
                    None
                    if epoch is None
                    else float(
                        epoch
                    )
                ),
            "eval_loss":
                float(
                    eval_loss
                ),
            "validation_chrfpp":
                float(
                    chrfpp
                ),
            "validation_bleu":
                float(
                    bleu
                ),
            "predictions_path":
                str(
                    predictions_path
                ),
            "evaluated_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
        }

        with open(
            metrics_path,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                metrics,
                f,
                ensure_ascii=False,
                indent=2,
            )

        return metrics

    finally:

        model = None

        clear_gpu_strong()


def select_checkpoint(
    checkpoint_df: pd.DataFrame,
) -> pd.Series:

    if checkpoint_df.empty:

        raise RuntimeError(
            "Tidak ada checkpoint yang dapat dipilih."
        )

    ranked = (
        checkpoint_df
        .sort_values(
            [
                "validation_chrfpp",
                "validation_bleu",
                "eval_loss",
                "epoch",
            ],
            ascending=[
                False,
                False,
                True,
                True,
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    return ranked.iloc[
        0
    ]


def materialize_selected_adapter(
    *,
    selected_checkpoint: Path,
    selected_adapter_dir: Path,
) -> None:

    if selected_adapter_dir.exists():

        shutil.rmtree(
            selected_adapter_dir
        )

    selected_adapter_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    required_candidates = [
        "adapter_config.json",
        "adapter_model.safetensors",
        "adapter_model.bin",
    ]

    copied = []

    for filename in required_candidates:

        source = (
            selected_checkpoint
            / filename
        )

        if source.exists():

            shutil.copy2(
                source,
                selected_adapter_dir
                / filename,
            )

            copied.append(
                filename
            )

    if (
        "adapter_config.json"
        not in copied
    ):

        raise FileNotFoundError(
            f"adapter_config.json tidak ditemukan "
            f"di {selected_checkpoint}"
        )

    if not (
        (
            selected_adapter_dir
            / "adapter_model.safetensors"
        ).exists()
        or (
            selected_adapter_dir
            / "adapter_model.bin"
        ).exists()
    ):

        raise FileNotFoundError(
            f"Bobot adapter tidak ditemukan "
            f"di {selected_checkpoint}"
        )

    tokenizer.save_pretrained(
        selected_adapter_dir
    )


def run_trial(
    *,
    rank: int,
    target_label: str,
    learning_rate: float,
    dropout: float,
    stage_name: str,
    stage_candidate: str,
) -> dict[str, Any]:

    payload = (
        canonical_trial_payload(
            rank=rank,
            target_label=target_label,
            learning_rate=learning_rate,
            dropout=dropout,
        )
    )

    fingerprint = (
        payload_fingerprint(
            payload
        )
    )

    trial_id = (
        f"trial_{fingerprint}"
    )

    trial_dir = (
        TRIAL_ROOT
        / trial_id
    )

    trial_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    completed_path = (
        trial_dir
        / "completed_summary.json"
    )

    failure_path = (
        trial_dir
        / "failure.json"
    )

    trainer_dir = (
        trial_dir
        / "trainer"
    )

    log_path = (
        trial_dir
        / "training_log.csv"
    )

    checkpoint_eval_root = (
        trial_dir
        / "checkpoint_validation"
    )

    selected_adapter_dir = (
        trial_dir
        / "selected_adapter"
    )

    selected_predictions_path = (
        trial_dir
        / "validation_predictions.csv"
    )

    checkpoint_metrics_path = (
        trial_dir
        / "checkpoint_metrics.csv"
    )

    # --------------------------------------------------------
    # Trial selesai -> reuse
    # --------------------------------------------------------

    if completed_path.exists():

        with open(
            completed_path,
            "r",
            encoding="utf-8",
        ) as f:

            completed = json.load(
                f
            )

        completed_payload = (
            completed.get(
                "payload",
                {}
            )
        )

        if (
            payload_fingerprint(
                completed_payload
            )
            != fingerprint
        ):

            raise RuntimeError(
                f"Summary trial tidak konsisten: {trial_id}"
            )

        print(
            f"{stage_name} / {stage_candidate}: "
            f"reuse {trial_id}"
        )

        result = dict(
            completed
        )

        result[
            "stage_name"
        ] = stage_name

        result[
            "stage_candidate"
        ] = stage_candidate

        return result

    model = None
    trainer = None

    training_seconds = float(
        "nan"
    )

    trainable = None
    total = None
    peak_training_gpu_gib = float(
        "nan"
    )

    try:

        print(
            "\n"
            + "=" * 80
        )

        print(
            f"{stage_name} — {stage_candidate}"
        )

        print(
            trial_id
        )

        print(
            json.dumps(
                payload,
                ensure_ascii=False,
                indent=2,
            )
        )

        # ====================================================
        # PHASE 1 — TRAIN 4 EPOCH
        # ====================================================

        model, trainable, total = (
            load_lora_training_model(
                payload
            )
        )

        torch.cuda.reset_peak_memory_stats(
            DEVICE_INDEX
        )

        training_args = TrainingArguments(
            output_dir=str(
                trainer_dir
            ),
            learning_rate=float(
                payload[
                    "learning_rate"
                ]
            ),
            lr_scheduler_type=(
                LR_SCHEDULER_TYPE
            ),
            warmup_ratio=(
                WARMUP_RATIO
            ),
            weight_decay=(
                WEIGHT_DECAY
            ),
            max_grad_norm=(
                MAX_GRAD_NORM
            ),
            per_device_train_batch_size=(
                PER_DEVICE_TRAIN_BATCH_SIZE
            ),
            per_device_eval_batch_size=(
                PER_DEVICE_EVAL_BATCH_SIZE
            ),
            gradient_accumulation_steps=(
                GRADIENT_ACCUMULATION_STEPS
            ),
            num_train_epochs=(
                MAX_EPOCHS
            ),
            bf16=True,
            fp16=False,

            # Loss validation tetap dihitung setiap epoch.
            eval_strategy="epoch",

            # Semua checkpoint akhir epoch disimpan.
            save_strategy="epoch",

            logging_strategy="steps",
            logging_steps=10,

            # V5: tidak memilih best model berdasarkan eval_loss.
            load_best_model_at_end=False,

            save_total_limit=(
                SAVE_TOTAL_LIMIT
            ),

            optim=(
                OPTIMIZER
            ),

            gradient_checkpointing=True,

            report_to=[],

            remove_unused_columns=False,

            dataloader_num_workers=0,

            seed=(
                TRAINING_SEED
            ),

            data_seed=(
                TRAINING_SEED
            ),
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=(
                train_dataset
            ),
            eval_dataset=(
                valid_dataset
            ),
            data_collator=(
                collator
            ),
            tokenizer=(
                tokenizer
            ),

            # V5: tidak menggunakan EarlyStoppingCallback.
            callbacks=[],
        )

        last_checkpoint = None

        if trainer_dir.exists():

            last_checkpoint = (
                get_last_checkpoint(
                    str(
                        trainer_dir
                    )
                )
            )

        if last_checkpoint:

            print(
                "Resume training:",
                last_checkpoint,
            )

        start = (
            time.perf_counter()
        )

        trainer.train(
            resume_from_checkpoint=(
                last_checkpoint
                if last_checkpoint
                else None
            )
        )

        training_seconds = (
            time.perf_counter()
            - start
        )

        peak_training_gpu_gib = float(
            torch.cuda.max_memory_allocated(
                DEVICE_INDEX
            )
            / (1024 ** 3)
        )

        history_df = pd.DataFrame(
            trainer.state.log_history
        )

        history_df.to_csv(
            log_path,
            index=False,
            encoding="utf-8-sig",
        )

        # ----------------------------------------------------
        # Ambil checkpoint yang benar-benar tersimpan
        # ----------------------------------------------------

        checkpoints = (
            list_epoch_checkpoints(
                trainer_dir
            )
        )

        if len(
            checkpoints
        ) != MAX_EPOCHS:

            raise RuntimeError(
                f"Diharapkan {MAX_EPOCHS} checkpoint akhir epoch, "
                f"tetapi ditemukan {len(checkpoints)}: "
                f"{[p.name for p in checkpoints]}"
            )

        print(
            "Checkpoint tersedia:",
            [
                path.name
                for path in checkpoints
            ],
        )

        # ----------------------------------------------------
        # Lepaskan training model SEBELUM evaluasi checkpoint.
        # ----------------------------------------------------

        try:

            if trainer is not None:

                trainer.optimizer = None
                trainer.lr_scheduler = None
                trainer.model = None

        except Exception:
            pass

        trainer = None
        model = None

        clear_gpu_strong()

        print(
            "VRAM free setelah training cleanup:",
            f"{free_vram_gib():.2f} GiB",
        )

        # ====================================================
        # PHASE 2 — EVALUASI SETIAP CHECKPOINT
        # ====================================================

        checkpoint_results = []

        for checkpoint_path in checkpoints:

            metrics = (
                evaluate_checkpoint(
                    trial_id=trial_id,
                    checkpoint_path=(
                        checkpoint_path
                    ),
                    history_df=(
                        history_df
                    ),
                    checkpoint_eval_root=(
                        checkpoint_eval_root
                    ),
                )
            )

            checkpoint_results.append(
                metrics
            )

        checkpoint_df = pd.DataFrame(
            checkpoint_results
        )

        checkpoint_df.to_csv(
            checkpoint_metrics_path,
            index=False,
            encoding="utf-8-sig",
        )

        selected_checkpoint_row = (
            select_checkpoint(
                checkpoint_df
            )
        )

        selected_checkpoint = Path(
            str(
                selected_checkpoint_row[
                    "checkpoint_path"
                ]
            )
        )

        selected_predictions_source = Path(
            str(
                selected_checkpoint_row[
                    "predictions_path"
                ]
            )
        )

        if not selected_predictions_source.exists():

            raise FileNotFoundError(
                f"Prediksi checkpoint terpilih tidak ditemukan: "
                f"{selected_predictions_source}"
            )

        shutil.copy2(
            selected_predictions_source,
            selected_predictions_path,
        )

        materialize_selected_adapter(
            selected_checkpoint=(
                selected_checkpoint
            ),
            selected_adapter_dir=(
                selected_adapter_dir
            ),
        )

        summary = {
            "trial_id":
                trial_id,
            "fingerprint":
                fingerprint,
            "completed_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
            "payload":
                payload,

            "checkpoint_selection_rule":
                CHECKPOINT_SELECTION_RULE,

            "selected_checkpoint":
                str(
                    selected_checkpoint
                ),

            "selected_checkpoint_name":
                str(
                    selected_checkpoint_row[
                        "checkpoint_name"
                    ]
                ),

            "selected_checkpoint_step":
                int(
                    selected_checkpoint_row[
                        "step"
                    ]
                ),

            "selected_checkpoint_epoch":
                (
                    None
                    if pd.isna(
                        selected_checkpoint_row[
                            "epoch"
                        ]
                    )
                    else float(
                        selected_checkpoint_row[
                            "epoch"
                        ]
                    )
                ),

            "selected_checkpoint_eval_loss":
                float(
                    selected_checkpoint_row[
                        "eval_loss"
                    ]
                ),

            "best_adapter_path":
                str(
                    selected_adapter_dir
                ),

            "validation_chrfpp":
                float(
                    selected_checkpoint_row[
                        "validation_chrfpp"
                    ]
                ),

            "validation_bleu":
                float(
                    selected_checkpoint_row[
                        "validation_bleu"
                    ]
                ),

            # Compatibility alias dengan notebook lama.
            "best_eval_loss":
                float(
                    selected_checkpoint_row[
                        "eval_loss"
                    ]
                ),

            "trainable_parameters":
                int(
                    trainable
                ),

            "total_parameters":
                int(
                    total
                ),

            "trainable_percent":
                float(
                    100
                    * trainable
                    / total
                ),

            "training_runtime_seconds":
                float(
                    training_seconds
                ),

            "peak_training_gpu_memory_gib":
                float(
                    peak_training_gpu_gib
                ),

            "checkpoint_metrics_path":
                str(
                    checkpoint_metrics_path
                ),

            "validation_predictions_path":
                str(
                    selected_predictions_path
                ),
        }

        with open(
            completed_path,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                summary,
                f,
                ensure_ascii=False,
                indent=2,
            )

        if failure_path.exists():

            failure_path.unlink()

        print(
            "\nCHECKPOINT TERPILIH"
        )

        display(
            checkpoint_df.sort_values(
                [
                    "validation_chrfpp",
                    "validation_bleu",
                    "eval_loss",
                ],
                ascending=[
                    False,
                    False,
                    True,
                ],
            )
        )

        print(
            "Selected:",
            selected_checkpoint.name,
        )

        print(
            "chrF++  :",
            summary[
                "validation_chrfpp"
            ],
        )

        print(
            "BLEU    :",
            summary[
                "validation_bleu"
            ],
        )

        print(
            "EvalLoss:",
            summary[
                "selected_checkpoint_eval_loss"
            ],
        )

        result = dict(
            summary
        )

        result[
            "stage_name"
        ] = stage_name

        result[
            "stage_candidate"
        ] = stage_candidate

        return result

    except Exception as exc:

        failure_info = {
            "trial_id":
                trial_id,
            "failed_at_utc":
                datetime.now(
                    timezone.utc
                ).isoformat(),
            "payload":
                payload,
            "error_type":
                type(
                    exc
                ).__name__,
            "error_message":
                str(
                    exc
                ),
            "traceback":
                traceback.format_exc(),
        }

        with open(
            failure_path,
            "w",
            encoding="utf-8",
        ) as f:

            json.dump(
                failure_info,
                f,
                ensure_ascii=False,
                indent=2,
            )

        raise

    finally:

        try:

            if trainer is not None:

                trainer.optimizer = None
                trainer.lr_scheduler = None
                trainer.model = None

        except Exception:
            pass

        trainer = None
        model = None

        clear_gpu_strong()

        print(
            "VRAM free akhir trial:",
            f"{free_vram_gib():.2f} GiB",
        )


def stage_dataframe(
    results: list[
        dict[str, Any]
    ],
) -> pd.DataFrame:

    rows = []

    for result in results:

        payload = (
            result[
                "payload"
            ]
        )

        rows.append(
            {
                "stage_name":
                    result[
                        "stage_name"
                    ],
                "stage_candidate":
                    result[
                        "stage_candidate"
                    ],
                "trial_id":
                    result[
                        "trial_id"
                    ],
                "rank":
                    int(
                        payload[
                            "rank"
                        ]
                    ),
                "alpha":
                    int(
                        payload[
                            "alpha"
                        ]
                    ),
                "target_label":
                    payload[
                        "target_label"
                    ],
                "target_modules":
                    ", ".join(
                        payload[
                            "target_modules"
                        ]
                    ),
                "learning_rate":
                    float(
                        payload[
                            "learning_rate"
                        ]
                    ),
                "dropout":
                    float(
                        payload[
                            "dropout"
                        ]
                    ),
                "selected_checkpoint_name":
                    result[
                        "selected_checkpoint_name"
                    ],
                "selected_checkpoint_epoch":
                    result[
                        "selected_checkpoint_epoch"
                    ],
                "selected_checkpoint_step":
                    int(
                        result[
                            "selected_checkpoint_step"
                        ]
                    ),
                "validation_chrfpp":
                    float(
                        result[
                            "validation_chrfpp"
                        ]
                    ),
                "validation_bleu":
                    float(
                        result[
                            "validation_bleu"
                        ]
                    ),
                "selected_checkpoint_eval_loss":
                    float(
                        result[
                            "selected_checkpoint_eval_loss"
                        ]
                    ),
                "trainable_parameters":
                    int(
                        result[
                            "trainable_parameters"
                        ]
                    ),
                "trainable_percent":
                    float(
                        result[
                            "trainable_percent"
                        ]
                    ),
                "training_runtime_seconds":
                    float(
                        result[
                            "training_runtime_seconds"
                        ]
                    ),
                "peak_training_gpu_memory_gib":
                    float(
                        result[
                            "peak_training_gpu_memory_gib"
                        ]
                    ),
                "best_adapter_path":
                    result[
                        "best_adapter_path"
                    ],
            }
        )

    return pd.DataFrame(
        rows
    )


def select_best_stage(
    dataframe: pd.DataFrame,
) -> pd.Series:

    if dataframe.empty:

        raise RuntimeError(
            "Stage tidak memiliki kandidat."
        )

    ranked = (
        dataframe
        .sort_values(
            by=[
                "validation_chrfpp",
                "validation_bleu",
                "selected_checkpoint_eval_loss",
                "trainable_parameters",
            ],
            ascending=[
                False,
                False,
                True,
                True,
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    return ranked.iloc[
        0
    ]


def save_stage(
    stage_name: str,
    dataframe: pd.DataFrame,
) -> pd.Series:

    output_path = (
        STAGE_ROOT
        / f"{stage_name}.csv"
    )

    dataframe.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    selected = (
        select_best_stage(
            dataframe
        )
    )

    selected_path = (
        STAGE_ROOT
        / f"{stage_name}_selected.json"
    )

    with open(
        selected_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            selected.to_dict(),
            f,
            ensure_ascii=False,
            indent=2,
            default=str,
        )

    display(
        dataframe.sort_values(
            [
                "validation_chrfpp",
                "validation_bleu",
                "selected_checkpoint_eval_loss",
                "trainable_parameters",
            ],
            ascending=[
                False,
                False,
                True,
                True,
            ],
        )
    )

    print(
        "Terpilih:",
        selected[
            "stage_candidate"
        ],
    )

    return selected


## 6A. Audit cache lokal sebelum training

Cell ini memastikan seluruh pemuatan backbone pada notebook sudah menggunakan cache `/workspace/.hf_home/hub` dan mode `local_files_only=True`. Jika cache tidak siap, training tidak dimulai.

In [7]:
# ============================================================
# CACHE-SAFE PRETRAIN AUDIT
# ============================================================

print("HF_HOME      :", HF_HOME)
print("HF_HUB_CACHE :", HF_HUB_CACHE)
print("MODEL CACHE  :", MODEL_CACHE_DIR)
print("CACHE GiB    :", f"{cache_size_gib:.2f}")
print("NETWORK LOAD : DISABLED untuk tokenizer/base model")
print("LOCAL ONLY   : True")

assert HF_HUB_CACHE == Path("/workspace/.hf_home/hub").resolve()
assert MODEL_CACHE_DIR.exists()
assert cache_size_gib >= 10.0

print("\nCACHE-SAFE AUDIT: PASS")


HF_HOME      : /workspace/.hf_home
HF_HUB_CACHE : /workspace/.hf_home/hub
MODEL CACHE  : /workspace/.hf_home/hub/models--meta-llama--Llama-3.1-8B-Instruct
CACHE GiB    : 29.93
NETWORK LOAD : DISABLED untuk tokenizer/base model
LOCAL ONLY   : True

CACHE-SAFE AUDIT: PASS


## 7. Train ulang hanya konfigurasi final V7

In [8]:
# ============================================================
# RETRAIN HANYA SATU KONFIGURASI FINAL V7
# ============================================================

LOCKED_FINAL = {
    "target_label": "L3",
    "rank": 16,
    "alpha": 32,
    "learning_rate": 2e-4,
    "dropout": 0.00,
    "selected_checkpoint_name": "checkpoint-159",
    "selected_checkpoint_step": 159,
    "selected_checkpoint_epoch": 3.0,
    "historical_validation_chrfpp": 34.885076358270545,
    "historical_validation_bleu": 9.882647808762755,
    "historical_validation_eval_loss": 1.8159797191619873,
    "historical_trial_id": "trial_45d7c6df7a120bfe",
    "historical_dataset_fingerprint":
        "6c8f3cb81c2dfe100390e26ca33147c8324e90dffa31092ae63c18310ba6fdcc",
}

if DATASET_FINGERPRINT != LOCKED_FINAL["historical_dataset_fingerprint"]:
    raise RuntimeError(
        "Fingerprint dataset tidak sama dengan dataset yang digunakan saat V7. "
        f"Current={DATASET_FINGERPRINT}, "
        f"V7={LOCKED_FINAL['historical_dataset_fingerprint']}. "
        "Jangan lanjutkan sebelum dataset dipastikan benar."
    )

RETRAIN_ROOT = (
    PROJECT_ROOT
    / "Hasil"
    / "final_training"
    / "lora_locked_retrain_v2"
)
TRIAL_ROOT = RETRAIN_ROOT / "trials"

RETRAIN_ROOT.mkdir(parents=True, exist_ok=True)
TRIAL_ROOT.mkdir(parents=True, exist_ok=True)

print("RETRAIN_ROOT :", RETRAIN_ROOT)
print("TRIAL_ROOT   :", TRIAL_ROOT)
print("\nKonfigurasi final yang akan dilatih ulang:")
print(json.dumps(LOCKED_FINAL, indent=2, ensure_ascii=False))

retrain_result = run_trial(
    rank=LOCKED_FINAL["rank"],
    target_label=LOCKED_FINAL["target_label"],
    learning_rate=LOCKED_FINAL["learning_rate"],
    dropout=LOCKED_FINAL["dropout"],
    stage_name="final_locked_retrain",
    stage_candidate="V7_FINAL_ONLY",
)

if retrain_result["trial_id"] != LOCKED_FINAL["historical_trial_id"]:
    raise RuntimeError(
        "Fingerprint trial berbeda dari V7 historis. "
        f"Current={retrain_result['trial_id']}, "
        f"V7={LOCKED_FINAL['historical_trial_id']}."
    )

print("\nSINGLE FINAL TRIAL: COMPLETE")
print("Trial ID:", retrain_result["trial_id"])


RETRAIN_ROOT : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_training/lora_locked_retrain_v2
TRIAL_ROOT   : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_training/lora_locked_retrain_v2/trials

Konfigurasi final yang akan dilatih ulang:
{
  "target_label": "L3",
  "rank": 16,
  "alpha": 32,
  "learning_rate": 0.0002,
  "dropout": 0.0,
  "selected_checkpoint_name": "checkpoint-159",
  "selected_checkpoint_step": 159,
  "selected_checkpoint_epoch": 3.0,
  "historical_validation_chrfpp": 34.885076358270545,
  "historical_validation_bleu": 9.882647808762755,
  "historical_validation_eval_loss": 1.8159797191619873,
  "historical_trial_id": "trial_45d7c6df7a120bfe",
  "historical_dataset_fingerprint": "6c8f3cb81c2dfe100390e26ca33147c8324e90dffa31092ae63c18310ba6fdcc"
}

final_locked_retrain — V7_FINAL_ONLY
trial_45d7c6df7a120bfe
{
  "model_id": "meta-llama/Llama-3.1-8B-Instruct",
  "dataset_fingerprint": "6c8f3cb81c2dfe100390e26ca33147c8324e90dffa31092ae63c18310ba6fdcc"

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Trainable: 41,943,040/8,072,204,288 (0.5196%)


/tmp/ipykernel_21521/3593521783.py:1346: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,1.549100,1.649087
2,0.866300,1.632938
3,0.128700,2.170694


Checkpoint tersedia: ['checkpoint-53', 'checkpoint-106', 'checkpoint-159', 'checkpoint-208']
VRAM free setelah training cleanup: 22.86 GiB

Evaluasi: trial_45d7c6df7a120bfe checkpoint-53


... (11 outputs left out)

Validation trial_45d7c6df7a120bfe_checkpoint-208:   0%|          | 0/53 [00:00<?, ?it/s]


CHECKPOINT TERPILIH


,trial_id,checkpoint_name,checkpoint_path,step,epoch,eval_loss,validation_chrfpp,validation_bleu,predictions_path,evaluated_at_utc
2,trial_45d7c6df7a120bfe,checkpoint-159,/workspace/10122233_CikitaMeydilvira_Skripsi/H...,159,3.000000,1.860979,34.786652,11.057929,/workspace/10122233_CikitaMeydilvira_Skripsi/H...,2026-08-13T17:34:18.477598+00:00
1,trial_45d7c6df7a120bfe,checkpoint-106,/workspace/10122233_CikitaMeydilvira_Skripsi/H...,106,2.000000,1.632938,31.300612,7.435509,/workspace/10122233_CikitaMeydilvira_Skripsi/H...,2026-08-13T17:32:26.180732+00:00
3,trial_45d7c6df7a120bfe,checkpoint-208,/workspace/10122233_CikitaMeydilvira_Skripsi/H...,208,3.940048,2.170694,30.675549,6.283661,/workspace/10122233_CikitaMeydilvira_Skripsi/H...,2026-08-13T17:37:00.352043+00:00
0,trial_45d7c6df7a120bfe,checkpoint-53,/workspace/10122233_CikitaMeydilvira_Skripsi/H...,53,1.000000,1.649087,25.951256,4.345455,/workspace/10122233_CikitaMeydilvira_Skripsi/H...,2026-08-13T17:29:52.526729+00:00


Selected: checkpoint-159
chrF++  : 34.78665232664383
BLEU    : 11.057929303516637
EvalLoss: 1.860978603363037
VRAM free akhir trial: 22.86 GiB

SINGLE FINAL TRIAL: COMPLETE
Trial ID: trial_45d7c6df7a120bfe


## 8. Paksa materialisasi checkpoint yang telah dikunci oleh V7

In [9]:
# ============================================================
# MATERIALIZE CHECKPOINT V7 YANG SUDAH TERKUNCI: checkpoint-159
# ============================================================

trial_id = retrain_result["trial_id"]
trial_dir = TRIAL_ROOT / trial_id
trainer_dir = trial_dir / "trainer"
checkpoint_metrics_path = trial_dir / "checkpoint_metrics.csv"

LOCKED_CHECKPOINT = (
    trainer_dir
    / LOCKED_FINAL["selected_checkpoint_name"]
)

if not LOCKED_CHECKPOINT.exists():
    available = [p.name for p in list_epoch_checkpoints(trainer_dir)]
    raise FileNotFoundError(
        f"{LOCKED_FINAL['selected_checkpoint_name']} tidak ditemukan. "
        f"Checkpoint tersedia: {available}"
    )

if not checkpoint_metrics_path.exists():
    raise FileNotFoundError(
        f"checkpoint_metrics.csv tidak ditemukan: {checkpoint_metrics_path}"
    )

checkpoint_df = pd.read_csv(checkpoint_metrics_path)

locked_rows = checkpoint_df[
    checkpoint_df["checkpoint_name"].astype(str)
    == LOCKED_FINAL["selected_checkpoint_name"]
].copy()

if len(locked_rows) != 1:
    raise RuntimeError(
        "Harus ada tepat satu baris metric untuk checkpoint-159."
    )

locked_row = locked_rows.iloc[0]

if int(locked_row["step"]) != LOCKED_FINAL["selected_checkpoint_step"]:
    raise RuntimeError(
        f"Step checkpoint berubah: {locked_row['step']}"
    )

locked_epoch = float(locked_row["epoch"])
if not np.isclose(
    locked_epoch,
    LOCKED_FINAL["selected_checkpoint_epoch"],
    atol=1e-9,
    rtol=0,
):
    raise RuntimeError(
        f"Epoch checkpoint berubah: {locked_epoch}"
    )

LOCKED_ADAPTER_DIR = (
    PROJECT_ROOT
    / "Artifacts"
    / "lora_locked_adapter"
)

materialize_selected_adapter(
    selected_checkpoint=LOCKED_CHECKPOINT,
    selected_adapter_dir=LOCKED_ADAPTER_DIR,
)

if not (LOCKED_ADAPTER_DIR / "adapter_config.json").exists():
    raise RuntimeError("adapter_config.json tidak berhasil dimaterialisasi.")

if not (
    (LOCKED_ADAPTER_DIR / "adapter_model.safetensors").exists()
    or (LOCKED_ADAPTER_DIR / "adapter_model.bin").exists()
):
    raise RuntimeError("Bobot adapter final tidak berhasil dimaterialisasi.")

print("LOCKED CHECKPOINT :", LOCKED_CHECKPOINT)
print("LOCKED ADAPTER    :", LOCKED_ADAPTER_DIR)
print("Current chrF++    :", float(locked_row["validation_chrfpp"]))
print("Current BLEU      :", float(locked_row["validation_bleu"]))
print("Current eval loss :", float(locked_row["eval_loss"]))


LOCKED CHECKPOINT : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_training/lora_locked_retrain_v2/trials/trial_45d7c6df7a120bfe/trainer/checkpoint-159
LOCKED ADAPTER    : /workspace/10122233_CikitaMeydilvira_Skripsi/Artifacts/lora_locked_adapter
Current chrF++    : 34.78665232664383
Current BLEU      : 11.057929303516635
Current eval loss : 1.860978603363037


## 9. Tulis `lora_locked_config.json` untuk 04C

In [10]:
# ============================================================
# TULIS CONFIG LOCK KOMPATIBEL DENGAN 04C
# ============================================================

CONFIG_LOCK_PATH = (
    PROJECT_ROOT
    / "Config"
    / "lora_locked_config.json"
)

CANONICAL_SELECTED_PATH = (
    PROJECT_ROOT
    / "Hasil"
    / "config_selection"
    / "lora"
    / "selected_lora_config.json"
)

RETRAIN_CONFIG_PATH = (
    RETRAIN_ROOT
    / "retrained_lora_locked_config.json"
)

for path in [
    CONFIG_LOCK_PATH.parent,
    CANONICAL_SELECTED_PATH.parent,
    RETRAIN_CONFIG_PATH.parent,
]:
    path.mkdir(parents=True, exist_ok=True)

locked_config = {
    "selected_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection_origin":
        "Recovered from completed FINAL V7 sequential-search evidence; "
        "this notebook retrains only the already-locked configuration.",
    "selection_method":
        "controlled target-first sequential search: "
        "target_modules -> rank -> learning_rate -> dropout; "
        "checkpoint selected by validation chrF++ in original V7",
    "scope_statement":
        "Best configuration within the tested V7 sequential search path; "
        "not a claim of global LoRA optimum.",
    "model_id": MODEL_ID,
    "config_code": "V7_FINAL",

    "rank": 16,
    "r": 16,
    "alpha": 32,
    "alpha_over_r": 2.0,
    "dropout": 0.0,
    "target_label": "L3",
    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    "learning_rate": 0.0002,

    "lr_scheduler_type": LR_SCHEDULER_TYPE,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "max_grad_norm": MAX_GRAD_NORM,
    "per_device_train_batch_size": PER_DEVICE_TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "max_epochs": MAX_EPOCHS,
    "early_stopping": False,
    "max_sequence_length": MAX_SEQUENCE_LENGTH,
    "assistant_only_loss": True,
    "training_seed": TRAINING_SEED,

    "final_trial_id": trial_id,
    "selected_checkpoint": str(LOCKED_CHECKPOINT),
    "best_checkpoint": str(LOCKED_CHECKPOINT),
    "selected_checkpoint_name": "checkpoint-159",
    "selected_checkpoint_epoch": 3.0,
    "selected_checkpoint_step": 159,
    "selected_checkpoint_eval_loss": float(locked_row["eval_loss"]),
    "best_eval_loss": float(locked_row["eval_loss"]),

    "best_adapter_path": str(LOCKED_ADAPTER_DIR),
    "locked_adapter_path": str(LOCKED_ADAPTER_DIR),

    "validation_chrfpp": float(locked_row["validation_chrfpp"]),
    "validation_bleu": float(locked_row["validation_bleu"]),

    "original_v7_selection_evidence": {
        "trial_id": LOCKED_FINAL["historical_trial_id"],
        "dataset_fingerprint":
            LOCKED_FINAL["historical_dataset_fingerprint"],
        "selected_checkpoint_name": "checkpoint-159",
        "selected_checkpoint_epoch": 3.0,
        "validation_chrfpp":
            LOCKED_FINAL["historical_validation_chrfpp"],
        "validation_bleu":
            LOCKED_FINAL["historical_validation_bleu"],
        "selected_checkpoint_eval_loss":
            LOCKED_FINAL["historical_validation_eval_loss"],
    },

    "trainable_parameters": int(retrain_result["trainable_parameters"]),
    "trainable_percent": float(retrain_result["trainable_percent"]),
    "training_runtime_seconds":
        float(retrain_result["training_runtime_seconds"]),
    "peak_training_gpu_memory_gib":
        float(retrain_result["peak_training_gpu_memory_gib"]),

    "dataset_fingerprint": DATASET_FINGERPRINT,
    "source_overlap_count": int(SOURCE_OVERLAP_COUNT),
    "exact_pair_overlap_count": int(EXACT_PAIR_OVERLAP_COUNT),
    "test_read": False,
    "train_rows": len(train_df),
    "validation_rows": len(valid_df),

    "retrain_artifact_root": str(RETRAIN_ROOT),
    "checkpoint_metrics_path": str(checkpoint_metrics_path),
    "reproduction_mode": "single_locked_config_retrain",
    "hyperparameter_search_rerun": False,
}

for path in [
    CONFIG_LOCK_PATH,
    CANONICAL_SELECTED_PATH,
    RETRAIN_CONFIG_PATH,
]:
    path.write_text(
        json.dumps(
            locked_config,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

print("Config lock        :", CONFIG_LOCK_PATH)
print("Canonical config   :", CANONICAL_SELECTED_PATH)
print("Retrain config     :", RETRAIN_CONFIG_PATH)
print("Locked adapter     :", LOCKED_ADAPTER_DIR)

print("\nCONFIG LOCK WRITTEN")


Config lock        : /workspace/10122233_CikitaMeydilvira_Skripsi/Config/lora_locked_config.json
Canonical config   : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/config_selection/lora/selected_lora_config.json
Retrain config     : /workspace/10122233_CikitaMeydilvira_Skripsi/Hasil/final_training/lora_locked_retrain_v2/retrained_lora_locked_config.json
Locked adapter     : /workspace/10122233_CikitaMeydilvira_Skripsi/Artifacts/lora_locked_adapter

CONFIG LOCK WRITTEN


## 10. Audit final

In [11]:
# ============================================================
# AUDIT FINAL SEBELUM MENJALANKAN 04C
# ============================================================

cfg = json.loads(
    CONFIG_LOCK_PATH.read_text(encoding="utf-8")
)

EXPECTED_TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

checks = {
    "model_id": cfg["model_id"] == "meta-llama/Llama-3.1-8B-Instruct",
    "config_code": cfg["config_code"] == "V7_FINAL",
    "r": int(cfg["r"]) == 16,
    "alpha": int(cfg["alpha"]) == 32,
    "dropout": np.isclose(float(cfg["dropout"]), 0.0),
    "learning_rate": np.isclose(float(cfg["learning_rate"]), 2e-4),
    "target_modules": list(cfg["target_modules"]) == EXPECTED_TARGET_MODULES,
    "checkpoint": cfg["selected_checkpoint_name"] == "checkpoint-159",
    "epoch": float(cfg["selected_checkpoint_epoch"]) == 3.0,
    "max_sequence_length": int(cfg["max_sequence_length"]) == 640,
    "train_rows": int(cfg["train_rows"]) == 417,
    "validation_rows": int(cfg["validation_rows"]) == 53,
    "test_read_false": cfg["test_read"] is False,
    "adapter_config":
        (LOCKED_ADAPTER_DIR / "adapter_config.json").exists(),
    "adapter_weights": (
        (LOCKED_ADAPTER_DIR / "adapter_model.safetensors").exists()
        or (LOCKED_ADAPTER_DIR / "adapter_model.bin").exists()
    ),
}

failed = [
    name
    for name, ok in checks.items()
    if not bool(ok)
]

display(
    pd.DataFrame(
        [{"check": k, "ok": bool(v)} for k, v in checks.items()]
    )
)

if failed:
    raise RuntimeError(
        f"AUDIT FINAL GAGAL: {failed}"
    )

print("\n" + "=" * 72)
print("LORA LOCKED ADAPTER REBUILD: COMPLETE")
print("=" * 72)
print("Adapter :", LOCKED_ADAPTER_DIR)
print("Config  :", CONFIG_LOCK_PATH)
print("chrF++  :", cfg["validation_chrfpp"])
print("BLEU    :", cfg["validation_bleu"])
print("EvalLoss:", cfg["selected_checkpoint_eval_loss"])
print("\nSelanjutnya: shutdown kernel notebook ini, cek nvidia-smi, lalu jalankan 04C.")


,check,ok
0,model_id,True
1,config_code,True
2,r,True
3,alpha,True
4,dropout,True
5,learning_rate,True
6,target_modules,True
7,checkpoint,True
8,epoch,True
9,max_sequence_length,True



LORA LOCKED ADAPTER REBUILD: COMPLETE
Adapter : /workspace/10122233_CikitaMeydilvira_Skripsi/Artifacts/lora_locked_adapter
Config  : /workspace/10122233_CikitaMeydilvira_Skripsi/Config/lora_locked_config.json
chrF++  : 34.78665232664383
BLEU    : 11.057929303516635
EvalLoss: 1.860978603363037

Selanjutnya: shutdown kernel notebook ini, cek nvidia-smi, lalu jalankan 04C.
